# Write the readout for desktop users

*Data Analysis in Python · Lesson 9 final exercise*

The head of e-commerce wants to know whether the new checkout works for desktop users, who never use in-app browsers. Run the whole analysis for them, and write a readout they can act on.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

Add your cells between the parameters cell and the **Check your work** cell. If you're stuck on a step, open its hint under the **Check your work** cell.

1. Read `ASSIGNMENTS_FILE` and `CONVERSIONS_FILE`, with the user IDs as text, and create the `users` table: one row per user in the test, with `converted` and `revenue`, 0 for users who didn't convert.
2. Create the `desktop_users` table: the users whose `device_category` is `desktop`. Every step after this one uses desktop users only.
3. Write the function `srm_p_value(counts)`, which returns the p-value of a chi-square test that the counts were split evenly, and create `srm_p`: its p-value for the `desktop_users` table's split. Read it before you go on.
4. Create `lift`: B's conversion rate lift over A, as a fraction.
5. Create `p_value`: the p-value of a two-sided two-proportion z-test of B's conversion rate against A's. Then create `ci_low` and `ci_high`: the 95% confidence interval of B's conversion rate minus A's, as fractions.
6. Create `users_needed`: the users per variant a test needs to detect `lift` from A's conversion rate, at a significance level of 0.05 and a power of 0.8, rounded up. Then create `weeks_needed`: the whole weeks a test would need to reach it, at the desktop users per variant per week that this `TEST_WEEKS`-week test had, rounded up.
7. Add a text cell for the head of e-commerce: whether the split holds, the lift with its interval, whether it's significant, and what they should do next, with the weeks it would take.
8. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: The `users` table has 80,054 rows, and the `desktop_users` table 27,856. `srm_p` is about `0.465`. `lift` is about `0.0391`. `p_value` is about `0.4614`, `ci_low` about `-0.0032` and `ci_high` about `0.0071`. `users_needed` is `201507`, and `weeks_needed` is `87`.

## Parameters

In [ ]:
import math

import pandas as pd
from scipy import stats
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import confint_proportions_2indep, proportion_effectsize, proportions_ztest

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
ASSIGNMENTS_FILE = DATA_URL + "ab_test_assignments.csv"
CONVERSIONS_FILE = DATA_URL + "ab_test_conversions.csv"
TEST_WEEKS = 6

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables, calls your function and reads the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    def number(value):
        return pd.api.types.is_number(value) and not isinstance(value, (pd.DataFrame, pd.Series))

    def near(value, *targets, tolerance=0.000001):
        return number(value) and any(abs(value - target) < tolerance for target in targets)

    users = found.get("users")
    desktop = found.get("desktop_users")
    if not isinstance(users, pd.DataFrame) or not {"variant", "converted", "device_category"} <= set(users.columns):
        results.append((False, "users should be the assignments with converted and revenue: a left join of the conversions, as in topic 9.1."))
    elif len(users) == 3851:
        results.append((False, "The users table has 3,851 rows: only the converters. Join with how=\"left\" to keep every assigned user."))
    elif len(users) != 80054:
        results.append((False, f"The users table should have one row per assigned user, 80,054. It has {len(users):,}."))
    elif users["converted"].sum() != 3851:
        results.append((False, f"converted should be True for the 3,851 users with a conversion. It's True for {users['converted'].sum():,}."))
    elif not isinstance(desktop, pd.DataFrame) or "variant" not in desktop.columns:
        results.append((False, "desktop_users should be the users table's rows whose device_category is desktop."))
    elif len(desktop) == 80054:
        results.append((False, "The desktop_users table has all 80,054 users. Filter on device_category == \"desktop\"."))
    elif len(desktop) != 27856:
        results.append((False, f"The desktop_users table should have the 27,856 desktop users. It has {len(desktop):,}."))
    else:
        results.append((True, f"80,054 users in the test, {len(desktop):,} of them on desktop, {desktop['converted'].sum():,} of whom converted."))

    srm_p = found.get("srm_p")
    if "srm_p" not in found:
        results.append((False, "There's no srm_p: check the split before you read the result. A segment can have its own mismatch. srm_p = srm_p_value(desktop_users[\"variant\"].value_counts())."))
    elif not number(srm_p):
        results.append((False, f"srm_p should be a number: the chi-square p-value of the desktop users' split. It is {srm_p!r}."))
    elif srm_p < 1e-25:
        results.append((False, f"srm_p is {srm_p:.1e}: that's all users' split. Check the desktop users' split: the readout is about them."))
    elif not near(srm_p, 0.464797):
        results.append((False, f"srm_p should be the chi-square p-value for 13,867 and 13,989 desktop users: about 0.465. It is {srm_p:.4g}."))
    else:
        results.append((True, f"The desktop split holds: 13,867 in A and 13,989 in B, p = {srm_p:.3f}, far above 0.001."))

    lift = found.get("lift")
    if not number(lift):
        results.append((False, f"lift should be a number: B's conversion rate divided by A's, minus 1, for desktop users. It is {lift!r}."))
    elif near(lift, 0.108668):
        results.append((False, "lift is 0.1087: all users' lift. Use the desktop_users table."))
    elif near(lift, -0.037633):
        results.append((False, "lift is -0.0376: that's A relative to B. Lift is B / A - 1."))
    elif not near(lift, 0.039104):
        results.append((False, f"lift should be B's conversion rate divided by A's, minus 1, for desktop users: about 0.0391. It is {lift:.4f}."))
    else:
        results.append((True, f"On desktop, B converts {lift:+.1%} relative to A: 5.13% against 4.93%."))

    p_value, low, high = found.get("p_value"), found.get("ci_low"), found.get("ci_high")
    if not number(p_value):
        results.append((False, f"p_value should be a number: the p-value from proportions_ztest() on the desktop users. It is {p_value!r}."))
    elif near(p_value, 0.230718):
        results.append((False, "p_value is 0.2307: a one-sided test, half the two-sided p-value. Choosing one side after seeing the result isn't a fair test, and a checkout change can make things worse: use the default, two-sided."))
    elif near(p_value, 0.001031):
        results.append((False, "p_value is 0.0010: that's all users. Test the desktop users."))
    elif not near(p_value, 0.461437):
        results.append((False, f"p_value should be the two-sided z-test's p-value for 717 of 13,989 against 684 of 13,867: about 0.4614. It is {p_value:.4g}."))
    elif not number(low) or not number(high):
        results.append((False, "ci_low and ci_high should be numbers: ci_low, ci_high = confint_proportions_2indep(conversions_b, users_b, conversions_a, users_a)."))
    elif near(low, -0.0070668, -0.0070614, tolerance=0.0000002):
        results.append((False, "ci_low and ci_high are A's rate minus B's. Put B first, so positive values mean B converts more."))
    elif not (near(low, -0.0032090, -0.0032038, tolerance=0.0000002) and near(high, 0.0070668, 0.0070614, tolerance=0.0000002)):
        results.append((False, f"ci_low and ci_high should be the 95% interval of B's rate minus A's for desktop users: about -0.0032 and 0.0071. They are {low:.4f} and {high:.4f}."))
    else:
        results.append((True, f"p = {p_value:.2f}, not significant. B's rate minus A's is between {low * 100:+.2f} and {high * 100:+.2f} percentage points: from {low / 0.049326:+.0%} to {high / 0.049326:+.0%} relative to A."))

    needed, weeks = found.get("users_needed"), found.get("weeks_needed")
    if not number(needed):
        results.append((False, f"users_needed should be a number: math.ceil(NormalIndPower().solve_power(effect_size=..., alpha=0.05, power=0.8)). It is {needed!r}."))
    elif near(needed, 201506.247, tolerance=0.01):
        results.append((False, "users_needed is 201,506.25: round it up with math.ceil(), to 201,507."))
    elif needed in (403013, 403014):
        results.append((False, f"users_needed is {needed:,}: both variants together. The brief asks for users per variant."))
    elif needed in (158727, 158728):
        results.append((False, "users_needed is 158,727: that's for a one-sided test. Plan for the two-sided test you ran."))
    elif not (needed == 201507 or needed in (201521, 201522)):
        results.append((False, f"users_needed should be the users per variant to detect a +3.9% lift from 4.93%, at alpha 0.05 and power 0.8, rounded up: 201,507. It is {needed!r}."))
    elif not number(weeks):
        results.append((False, f"weeks_needed should be a number: math.ceil(users_needed / (len(desktop_users) / 2 / TEST_WEEKS)). It is {weeks!r}."))
    elif near(weeks, 86.8066, 86.8125, tolerance=0.01):
        results.append((False, f"weeks_needed is {weeks:.2f}: round it up to whole weeks with math.ceil()."))
    elif weeks in (31, 44):
        results.append((False, f"weeks_needed is {weeks}: {'that uses all users per week, not desktop users' if weeks == 31 else 'users_per_week counts both variants: divide by 2 as well'}."))
    elif weeks != 87:
        results.append((False, f"weeks_needed should be users_needed divided by the desktop users per variant per week, 2,321.33, rounded up: 87. It is {weeks!r}."))
    else:
        results.append((True, f"Detecting +3.9% on desktop takes {needed:,} users per variant: {weeks} weeks of desktop traffic, against the test's {found.get('TEST_WEEKS', 6)}."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run. Read your text cell again: it's what the head of e-commerce will read.")


check_your_work()

## Hints

Stuck on a step? Open its hint. Each one says where the course showed it, and gives the shape of the code, with `...` where your part goes. Try the step from the hint before you open the solution.

<details>
<summary><b>Step 1: one row per user</b></summary>

Topic 9.1.: read the IDs as text, keep every assigned user with a left join, and give the users without a conversion $0.

~~~python
assignments = pd.read_csv(ASSIGNMENTS_FILE, dtype={"user_pseudo_id": str}, parse_dates=["assigned_at"])
conversions = pd.read_csv(...)
users = assignments.merge(conversions[["user_pseudo_id", "revenue"]], on=..., how=..., validate="one_to_one")
users["converted"] = ...
users["revenue"] = ...
~~~

</details>

<details>
<summary><b>Step 2: desktop users</b></summary>

Lesson 2.: a filter on `device_category`.

~~~python
desktop_users = users[users[...] == ...]
len(desktop_users)
~~~

</details>

<details>
<summary><b>Step 3: the split</b></summary>

Topic 9.2.: the chi-square test on the number of users in each variant. Below 0.001, stop and find the cause before you read anything else.

~~~python
def srm_p_value(counts):
    """..."""
    return ...


srm_p = srm_p_value(desktop_users[...].value_counts())
~~~

</details>

<details>
<summary><b>Step 4: the lift</b></summary>

Topic 9.1.: B relative to A, the control.

~~~python
rates = desktop_users.groupby("variant")[...].mean()
lift = ...
~~~

</details>

<details>
<summary><b>Step 5: the test and the interval</b></summary>

Topic 9.4.: conversions and users per variant, B first. Two-sided is the default: leave it.

~~~python
counts = desktop_users.groupby("variant")["converted"].agg(["sum", "count"])
z, p_value = proportions_ztest(counts.loc[["B", "A"], "sum"], counts.loc[["B", "A"], ...])
ci_low, ci_high = confint_proportions_2indep(counts.loc["B", "sum"], counts.loc["B", "count"], ..., ...)
~~~

</details>

<details>
<summary><b>Step 6: users and weeks</b></summary>

Topic 9.5.: the effect size from A's rate and A's rate plus the lift, the users per variant, then the desktop users per variant per week.

~~~python
effect = proportion_effectsize(rates["A"] * (1 + lift), rates["A"])
users_needed = math.ceil(NormalIndPower().solve_power(effect_size=effect, alpha=..., power=...))
weeks_needed = math.ceil(users_needed / (len(desktop_users) / 2 / ...))
~~~

</details>

<details>
<summary><b>Step 7: the text cell</b></summary>

Choose **Insert → Text cell**. Start with the split, because everything else depends on it. Give the lift with its interval as a relative range (divide the interval by A's rate), say what "not significant" does and doesn't mean here, and turn the weeks into a recommendation.

</details>

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
assignments = pd.read_csv(ASSIGNMENTS_FILE, dtype={"user_pseudo_id": str}, parse_dates=["assigned_at"])
conversions = pd.read_csv(CONVERSIONS_FILE, dtype={"user_pseudo_id": str}, parse_dates=["converted_at"])
users = assignments.merge(conversions[["user_pseudo_id", "revenue"]], on="user_pseudo_id", how="left", validate="one_to_one")
users["converted"] = users["revenue"].notna()
users["revenue"] = users["revenue"].fillna(0)
desktop_users = users[users["device_category"] == "desktop"]
len(users), len(desktop_users)
```

```python
def srm_p_value(counts):
    """Return the p-value of a chi-square test that the counts were split evenly between the variants."""
    return stats.chisquare(counts).pvalue


srm_p = srm_p_value(desktop_users["variant"].value_counts())
desktop_users["variant"].value_counts(), srm_p
```

```python
rates = desktop_users.groupby("variant")["converted"].mean()
lift = rates["B"] / rates["A"] - 1
counts = desktop_users.groupby("variant")["converted"].agg(["sum", "count"])
z, p_value = proportions_ztest(counts.loc[["B", "A"], "sum"], counts.loc[["B", "A"], "count"])
ci_low, ci_high = confint_proportions_2indep(counts.loc["B", "sum"], counts.loc["B", "count"], counts.loc["A", "sum"], counts.loc["A", "count"])
rates, lift, p_value, ci_low / rates["A"], ci_high / rates["A"]
```

```python
effect = proportion_effectsize(rates["A"] * (1 + lift), rates["A"])
users_needed = math.ceil(NormalIndPower().solve_power(effect_size=effect, alpha=0.05, power=0.8))
weeks_needed = math.ceil(users_needed / (len(desktop_users) / 2 / TEST_WEEKS))
users_needed, weeks_needed
```

```markdown
**Checkout redesign, desktop users.** 27,856 desktop users, September 1 to October 12, 2025.

- **The split holds.** 13,867 users saw the old checkout and 13,989 the new one (p = 0.465). Desktop has no in-app browsers, so the logging problem that broke the overall test doesn't touch it, and the two groups are comparable.
- **B converts 3.9% more, but the test can't tell that from chance.** 5.13% against 4.93% (p = 0.46). The true difference is somewhere between −0.32 and +0.71 percentage points: from about 7% worse to 14% better than the old checkout.
- **This test was never big enough.** Detecting a lift of 3.9% takes 201,507 desktop users per variant: 87 weeks at this test's desktop traffic. Not significant here means "couldn't tell", not "no effect".
- **Next step.** Don't call B a winner on desktop. If the redesign is cheaper to run or better for other reasons, ship it on that basis. To measure it, test on all devices once the in-app logging is fixed, and agree beforehand on the smallest lift worth detecting and the weeks that takes.
```

</details>